# Table I: runtime and PoE agreement

Compare plaintext PoE, the proposed consensus protocol, Nawaz-inspired HE-PoE, and Luo-inspired SS-PoE. Run cells in order from a fresh kernel with the complete project folder available.

SARCOS cases `(M, Ni)`: `(20, 4)`, `(20, 19)`, `(40, 4)`. Diabetes cases: `(10, 4)`, `(20, 4)`, `(20, 19)`. The comparison implementations and timing definitions are described below and in `README.md`.

In [ ]:
# Colab/Linux setup: install the missing Python dependency and build Lattigo.
import importlib.util
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import urllib.request


def is_project_root(path):
    path = Path(path)
    return (path / "table1.ipynb").is_file() and (
        path / "nawaz_sdsgp/lattigo_backend/main.go"
    ).is_file()


# Colab opens Drive notebooks with /content as cwd. Mount Drive and locate
# the uploaded project unless PPDGPR_PROJECT_ROOT was set explicitly.
project_root = Path(os.environ.get("PPDGPR_PROJECT_ROOT", Path.cwd())).expanduser()
if not is_project_root(project_root) and "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    drive_root = Path("/content/drive/MyDrive")
    candidates = sorted(
        {
            backend_dir.parents[1]
            for backend_dir in drive_root.rglob("lattigo_backend")
            if backend_dir.parent.name == "nawaz_sdsgp"
            and is_project_root(backend_dir.parents[1])
        }
    )
    if len(candidates) == 1:
        project_root = candidates[0]
    elif not candidates:
        raise FileNotFoundError(
            "Could not find the uploaded project under MyDrive. Set "
            "os.environ['PPDGPR_PROJECT_ROOT'] to its absolute Drive path."
        )
    else:
        raise RuntimeError(
            f"Multiple project copies found: {candidates}. Set "
            "os.environ['PPDGPR_PROJECT_ROOT'] to the one to use."
        )
if not is_project_root(project_root):
    raise FileNotFoundError(f"Not in the project folder: {project_root.resolve()}")
os.chdir(project_root)
print("Project root:", Path.cwd())

if importlib.util.find_spec("gpytorch") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gpytorch>=1.14,<2"],
        check=True,
    )

if os.name != "nt":
    backend_dir = Path("nawaz_sdsgp/lattigo_backend").resolve()
    # Build outside mounted Drive: DriveFS can reject/slow executable output.
    backend = (
        Path("/content/nawaz_lattigo")
        if "google.colab" in sys.modules
        else backend_dir / "nawaz_lattigo"
    )
    sources = [backend_dir / name for name in ("main.go", "go.mod", "go.sum")]
    rebuild = not backend.exists() or any(
        source.stat().st_mtime > backend.stat().st_mtime for source in sources
    )
    if rebuild:
        go = shutil.which("go")
        if go is None:
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "golang-go"], check=True)
            go = shutil.which("go")
        go_version_text = subprocess.run(
            [go, "version"], text=True, capture_output=True, check=True
        ).stdout
        version_token = next(
            token for token in go_version_text.split() if token.startswith("go1.")
        )
        go_version = tuple(int(part) for part in version_token[2:].split(".")[:2])
        if go_version < (1, 24):
            # Lattigo v6.2 requires Go >=1.24; old Colab images ship Go 1.18.
            toolchain_parent = Path("/content/go-modern")
            modern_go = toolchain_parent / "go/bin/go"
            if not modern_go.exists():
                with urllib.request.urlopen(
                    "https://go.dev/dl/?mode=json", timeout=60
                ) as response:
                    releases = json.load(response)
                release = next(item for item in releases if item.get("stable"))
                archive = next(
                    item
                    for item in release["files"]
                    if item["os"] == "linux"
                    and item["arch"] == "amd64"
                    and item["kind"] == "archive"
                )
                archive_path = Path("/content") / archive["filename"]
                urllib.request.urlretrieve(
                    "https://go.dev/dl/" + archive["filename"], archive_path
                )
                digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
                if digest != archive["sha256"]:
                    raise RuntimeError("Downloaded Go toolchain checksum mismatch")
                toolchain_parent.mkdir(parents=True, exist_ok=True)
                with tarfile.open(archive_path, "r:gz") as tar:
                    tar.extractall(toolchain_parent, filter="data")
                archive_path.unlink()
            go = str(modern_go)
            print(
                "Using downloaded toolchain:",
                subprocess.run(
                    [go, "version"], text=True, capture_output=True, check=True
                ).stdout.strip(),
            )
        build_env = dict(os.environ, GOTOOLCHAIN="auto")
        build = subprocess.run(
            [go, "build", "-o", str(backend.resolve()), "."],
            cwd=backend_dir,
            env=build_env,
            text=True,
            capture_output=True,
            check=False,
        )
        if build.returncode:
            go_version = subprocess.run(
                [go, "version"], text=True, capture_output=True
            ).stdout.strip()
            raise RuntimeError(
                f"Lattigo build failed ({go_version}).\n"
                f"stdout:\n{build.stdout}\nstderr:\n{build.stderr}"
            )
        backend.chmod(0o755)
    os.environ["NAWAZ_LATTIGO_BACKEND"] = str(backend.resolve())
    print("Lattigo backend:", backend.resolve())

In [ ]:
import gc
import platform
import time

import gpytorch
import numpy as np
import pandas as pd
import torch
from scipy.io import loadmat
from IPython.display import display
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from final_experiments import (
    benchmark_diabetes_local_prediction,
    benchmark_he_poe,
    benchmark_poe,
    benchmark_private_consensus,
    benchmark_sarcos_local_prediction,
    build_graph_and_weights,
    consensus_mean_variance,
    proposed_owner_hyperparameters,
)
from hypOpt import hyperparameter_optimization, hyperparameter_optimization_1D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("GPyTorch:", gpytorch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Configuration

Use 4,000 SARCOS training observations, 200 SARCOS test observations, 100 consensus iterations, and 20 timing repeats. `Lz_values=[10000]` sets the prediction scale to $L_z=10^{-4}$; hyperparameter optimization uses $2^{-20}$. Communication latency is 0.2 ms per modeled phase.

Setting `sarcos_training_samples=None` selects all 44,484 training observations and changes the experiment.

In [ ]:
seed = 1
sarcos_cases = [(20, 4), (20, 19), (40, 4)]
diabetes_cases = [(10, 4), (20, 4), (20, 19)]
cases = sarcos_cases
Lz_values = [10_000]

base_config = {
    "q": 2**40,
    "Lw": 1 / 40,
    "Lz": 1 / 2**20,  # Hyperparameter optimization; protocol uses Lz_values
    "Thyp": 30,
    "eta": 0.1,
    "decay": 0.99,
    "T": 100,
    "communication_latency": 0.0002,  # 0.2 ms per modeled communication delay
}
num_repeats = 20
# CPU protocol workers stay parallel; one model worker feeds a single GPU.
num_workers = min(4, os.cpu_count() or 1)
model_workers = 1 if device.type == "cuda" else num_workers
nawaz_communication_phases = 6
sarcos_test_samples = 200
sarcos_training_samples = 4000  # SARCOS training samples in the manuscript


np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print("SARCOS cases:", sarcos_cases)
print("Diabetes cases:", diabetes_cases)
print("Lz:", Lz_values)

## Data preparation

Load and shuffle each dataset once. Each topology receives the same ordered observations, repartitioned for its agent count. SARCOS uses a random training subset and the first 200 observations of the test file. Diabetes uses an 80/20 split with `random_state=42`.

In [ ]:
# SARCOS
sarcos_train = loadmat("data/sarcos_inv.mat")["sarcos_inv"]
sarcos_test = loadmat("data/sarcos_inv_test.mat")["sarcos_inv_test"]
sarcos_x_all = torch.tensor(sarcos_train[:, :21], dtype=torch.float32)
sarcos_y_all = torch.tensor(sarcos_train[:, 21:], dtype=torch.float32)
sarcos_x_test = torch.tensor(
    sarcos_test[:sarcos_test_samples, :21], dtype=torch.float32, device=device
)
sarcos_y_test = torch.tensor(
    sarcos_test[:sarcos_test_samples, 21:], dtype=torch.float32, device=device
)
sarcos_rng = np.random.default_rng(seed)
if sarcos_training_samples is None:
    sarcos_selection = np.arange(len(sarcos_x_all))
else:
    if not max(M for M, _ in cases) <= sarcos_training_samples <= len(sarcos_x_all):
        raise ValueError("Invalid sarcos_training_samples")
    sarcos_selection = sarcos_rng.choice(
        len(sarcos_x_all), size=int(sarcos_training_samples), replace=False
    )
sarcos_x_selected = sarcos_x_all[sarcos_selection]
sarcos_y_selected = sarcos_y_all[sarcos_selection]
sarcos_permutation = torch.randperm(
    len(sarcos_x_selected), generator=torch.Generator().manual_seed(seed)
)
sarcos_x_shuffled = sarcos_x_selected[sarcos_permutation]
sarcos_y_shuffled = sarcos_y_selected[sarcos_permutation]

# Diabetes: the same train/test split as figs2-3.ipynb.
diabetes = load_diabetes()
diab_x_train_np, diab_x_test_np, diab_y_train_np, diab_y_test_np = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)
diab_x_train = torch.tensor(diab_x_train_np, dtype=torch.float32)
diab_y_train = torch.tensor(diab_y_train_np, dtype=torch.float32)
diab_x_test = torch.tensor(diab_x_test_np, dtype=torch.float32, device=device)
diab_y_test = torch.tensor(diab_y_test_np, dtype=torch.float32, device=device)
diab_permutation = torch.randperm(
    len(diab_x_train), generator=torch.Generator().manual_seed(seed)
)
diab_x_shuffled = diab_x_train[diab_permutation]
diab_y_shuffled = diab_y_train[diab_permutation]

print("SARCOS selected train/test:", len(sarcos_x_shuffled), len(sarcos_x_test))
print("Diabetes train/test:", len(diab_x_shuffled), len(diab_x_test))

## Shared SARCOS and Diabetes cases

For each topology, optimize the local GP hyperparameters, compute plaintext PoE predictions, and run private consensus. Training time is recorded separately.

Online totals combine local prediction with PoE aggregation or private consensus. Measurements are added for each repeat before computing the mean and population standard deviation. HE-PoE is evaluated once per dataset and agent count, using the first matching topology.

In [ ]:
timing_rows = []
accuracy_rows = []
offline_rows = []
graph_rows = []
nawaz_build_rows = []
luo_sarcos_owner_hyperparameters = None
luo_diabetes_owner_hyperparameters = None
luo_sarcos_poe_reference = None
luo_diabetes_poe_reference = None
luo_sarcos_by_M = {}
luo_diabetes_by_M = {}
nawaz_sarcos_agent_counts = set()
nawaz_diabetes_agent_counts = set()


def append_timing(dataset, M, Ni, method, test_points, samples):
    samples = np.asarray(samples, dtype=float)
    timing_rows.append(
        {
            "dataset": dataset,
            "M": M,
            "Ni": Ni,
            "method": method,
            "test_points": int(test_points),
            "mean_total_seconds": samples.mean(),
            "std_total_seconds": samples.std(),
            "points_per_second": test_points / samples.mean(),
        }
    )


def run_and_record_nawaz(
    dataset,
    M,
    test_y,
    poe_mean,
    poe_variance,
    local_means,
    local_variances,
    local_prediction_times,
):
    result = benchmark_he_poe(
        local_means,
        local_variances,
        repeats=num_repeats,
        communication_latency=base_config["communication_latency"],
        workers=num_workers,
        communication_phases=nawaz_communication_phases,
        desc=f"{dataset} multiparty HE-PoE M={M}",
    )
    prediction = result["prediction"]
    target = test_y.detach().cpu().numpy()
    error = prediction - target
    target_rmse = (
        np.sqrt(np.mean(error**2))
        if error.ndim == 1
        else np.sqrt(np.mean(np.sum(error**2, axis=1)))
    )
    protocol_error = prediction - poe_mean.detach().cpu().numpy()
    protocol_rmse = (
        np.sqrt(np.mean(protocol_error**2))
        if protocol_error.ndim == 1
        else np.sqrt(np.mean(np.sum(protocol_error**2, axis=1)))
    )
    variance_protocol_error = result["variance"] - poe_variance.detach().cpu().numpy()
    variance_protocol_rmse = (
        np.sqrt(np.mean(variance_protocol_error**2))
        if variance_protocol_error.ndim == 1
        else np.sqrt(np.mean(np.sum(variance_protocol_error**2, axis=1)))
    )
    total_times = np.asarray(local_prediction_times) + result["protocol_times"]
    append_timing(
        dataset,
        M,
        np.nan,
        "Nawaz-inspired HE-PoE",
        len(target),
        total_times,
    )
    accuracy_rows.append(
        {
            "dataset": dataset,
            "M": M,
            "Ni": np.nan,
            "method": "Nawaz-inspired HE-PoE",
            "prediction_rmse": target_rmse,
            "poe_agreement_rmse": protocol_rmse,
            "poe_variance_agreement_rmse": variance_protocol_rmse,
        }
    )
    nawaz_build_rows.append(
        {
            "dataset": dataset,
            "M": M,
            "Ni": np.nan,
            "local_exact_gp_prediction_seconds_included": np.mean(
                local_prediction_times
            ),
            "he_seconds_included": result["he_times"].mean(),
            "agent_division_seconds_included": result["division_times"].mean(),
            "empirical_latency_seconds_included": result["empirical_latency_time"],
            "hyperparameter_source": "Same proposed local GP models",
            "kernel": "Scaled isotropic RBF",
            "packed_values_per_agent": result["packed_values_per_agent"],
            "ciphertexts_per_agent": result["ciphertexts_per_agent"],
            "authorized_decrypting_agents": M,
            "maximum_secure_sum_error": result["maximum_secure_sum_error"],
        }
    )
    print(
        f"{dataset} HE-PoE M={M}: target RMSE={target_rmse:.6f}; "
        f"HE-to-plaintext PoE mean RMSE={protocol_rmse:.12e}; "
        f"variance RMSE={variance_protocol_rmse:.12e}; "
        f"total={total_times.mean():.6f} +/- {total_times.std():.6f} s"
    )
    return result


for case_index, (M, Ni) in enumerate(cases, 1):
    run_diabetes = (M, Ni) in diabetes_cases
    print(f'\n{"="*72}\nCase {case_index}/{len(cases)}: M={M}, Ni={Ni}\n{"="*72}')
    config = dict(base_config)
    config.update({"M": M, "Ni": Ni, "k": sarcos_y_shuffled.shape[1]})
    graph, wbar = build_graph_and_weights(M, Ni, config["Lw"], seed)
    config["Wbar"] = wbar
    graph_rows.append(
        {
            "M": M,
            "Ni": Ni,
            "edges": graph.number_of_edges(),
            "connected": True,
            "assumption_2": True,
            "Wbar_integer": True,
        }
    )

    sarcos_x_parts = [x.to(device) for x in torch.tensor_split(sarcos_x_shuffled, M)]
    sarcos_y_parts = [y.to(device) for y in torch.tensor_split(sarcos_y_shuffled, M)]
    diab_x_parts = [x.to(device) for x in torch.tensor_split(diab_x_shuffled, M)]
    diab_y_parts = [
        y.unsqueeze(-1).to(device) for y in torch.tensor_split(diab_y_shuffled, M)
    ]

    # Offline training: performed for correctness, excluded from online totals.
    hyper_start = time.perf_counter()
    sarcos_models, sarcos_likelihoods, _, _ = hyperparameter_optimization(
        sarcos_x_parts,
        sarcos_y_parts,
        graph,
        config,
        [10.0, 10.0, 10.0],
        device,
        show_progress=True,
        num_workers=model_workers,
    )
    sarcos_hyper_time = time.perf_counter() - hyper_start
    for agent in range(M):
        for output in range(config["k"]):
            sarcos_models[agent][output] = sarcos_models[agent][output].to(device)
            sarcos_likelihoods[agent][output] = sarcos_likelihoods[agent][output].to(
                device
            )

    offline_rows.append(
        {
            "dataset": "SARCOS",
            "M": M,
            "Ni": Ni,
            "hyperparameter_seconds_excluded": sarcos_hyper_time,
        }
    )
    if run_diabetes:
        hyper_start = time.perf_counter()
        diab_models, diab_likelihoods, _, _, _ = hyperparameter_optimization_1D(
            diab_x_parts,
            diab_y_parts,
            graph,
            config,
            [10.0, 10.0, 20.0],
            device,
            progress_desc=f"Diabetes M={M}, Ni={Ni} hyperparameters",
            show_progress=True,
        )
        diab_hyper_time = time.perf_counter() - hyper_start
        offline_rows.append(
            {
                "dataset": "Diabetes",
                "M": M,
                "Ni": Ni,
                "hyperparameter_seconds_excluded": diab_hyper_time,
            }
        )

    # SARCOS online pipeline
    sarcos_means, sarcos_vars, sarcos_local_times = benchmark_sarcos_local_prediction(
        sarcos_models,
        sarcos_likelihoods,
        sarcos_x_test,
        num_repeats,
        device,
        workers=model_workers,
    )
    sarcos_poe_mean, sarcos_poe_var, sarcos_poe_times = benchmark_poe(
        sarcos_means, sarcos_vars, num_repeats, "SARCOS PoE aggregation"
    )
    if (M, Ni) == (20, 4):
        luo_sarcos_poe_reference = sarcos_poe_mean.detach().cpu().clone()
    append_timing(
        "SARCOS",
        M,
        Ni,
        "PoE baseline",
        sarcos_test_samples,
        sarcos_local_times + sarcos_poe_times,
    )
    sarcos_baseline_rmse = torch.sqrt(
        torch.mean(torch.sum((sarcos_poe_mean - sarcos_y_test) ** 2, dim=1))
    ).item()
    accuracy_rows.append(
        {
            "dataset": "SARCOS",
            "M": M,
            "Ni": Ni,
            "method": "PoE baseline",
            "prediction_rmse": sarcos_baseline_rmse,
            "poe_agreement_rmse": 0.0,
            "poe_variance_agreement_rmse": 0.0,
        }
    )
    sarcos_private_times, sarcos_final = benchmark_private_consensus(
        sarcos_means,
        sarcos_vars,
        graph,
        config,
        Lz_values,
        num_repeats,
        device,
        "SARCOS private consensus",
    )
    for lz_index, Lz in enumerate(Lz_values):
        append_timing(
            "SARCOS",
            M,
            Ni,
            f"Proposed Lz={Lz}",
            sarcos_test_samples,
            sarcos_local_times + sarcos_private_times[:, lz_index],
        )
        private_mean, private_var = consensus_mean_variance(sarcos_final[lz_index])
        prediction_rmse = (
            torch.sqrt(
                torch.mean(
                    torch.sum((private_mean - sarcos_y_test.unsqueeze(0)) ** 2, dim=2),
                    dim=1,
                )
            )
            .mean()
            .item()
        )
        agreement_rmse = (
            torch.sqrt(
                torch.mean(
                    torch.sum(
                        (private_mean - sarcos_poe_mean.unsqueeze(0)) ** 2, dim=2
                    ),
                    dim=1,
                )
            )
            .mean()
            .item()
        )
        variance_agreement_rmse = (
            torch.sqrt(
                torch.mean(
                    torch.sum((private_var - sarcos_poe_var.unsqueeze(0)) ** 2, dim=2),
                    dim=1,
                )
            )
            .mean()
            .item()
        )
        accuracy_rows.append(
            {
                "dataset": "SARCOS",
                "M": M,
                "Ni": Ni,
                "method": f"Proposed Lz={Lz}",
                "prediction_rmse": prediction_rmse,
                "poe_agreement_rmse": agreement_rmse,
                "poe_variance_agreement_rmse": variance_agreement_rmse,
            }
        )

    # Nawaz depends on M, not the proposed neighbor graph: run once per M.
    if M not in nawaz_sarcos_agent_counts:
        run_and_record_nawaz(
            "SARCOS",
            M,
            sarcos_y_test,
            sarcos_poe_mean,
            sarcos_poe_var,
            sarcos_means,
            sarcos_vars,
            sarcos_local_times,
        )
        nawaz_sarcos_agent_counts.add(M)
        owner_hyperparameters = proposed_owner_hyperparameters(
            sarcos_models, sarcos_likelihoods
        )
        luo_sarcos_by_M[M] = {
            "owner_hyperparameters": owner_hyperparameters,
            "poe_reference": sarcos_poe_mean.detach().cpu().clone(),
            "poe_variance_reference": sarcos_poe_var.detach().cpu().clone(),
        }
        if M == 20:
            luo_sarcos_owner_hyperparameters = owner_hyperparameters

    if not run_diabetes:
        del sarcos_models, sarcos_likelihoods
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        continue

    # Diabetes online pipeline
    diab_means, diab_vars, diab_local_times = benchmark_diabetes_local_prediction(
        diab_models,
        diab_likelihoods,
        diab_x_test,
        num_repeats,
        device,
        workers=model_workers,
    )
    diab_poe_mean, diab_poe_var, diab_poe_times = benchmark_poe(
        diab_means, diab_vars, num_repeats, "Diabetes PoE aggregation"
    )
    if (M, Ni) == (20, 4):
        luo_diabetes_poe_reference = diab_poe_mean.detach().cpu().clone()
    append_timing(
        "Diabetes",
        M,
        Ni,
        "PoE baseline",
        len(diab_x_test),
        diab_local_times + diab_poe_times,
    )
    diab_baseline_rmse = torch.sqrt(
        torch.mean((diab_poe_mean - diab_y_test) ** 2)
    ).item()
    accuracy_rows.append(
        {
            "dataset": "Diabetes",
            "M": M,
            "Ni": Ni,
            "method": "PoE baseline",
            "prediction_rmse": diab_baseline_rmse,
            "poe_agreement_rmse": 0.0,
            "poe_variance_agreement_rmse": 0.0,
        }
    )
    diab_private_times, diab_final = benchmark_private_consensus(
        diab_means,
        diab_vars,
        graph,
        config,
        Lz_values,
        num_repeats,
        device,
        "Diabetes private consensus",
    )
    for lz_index, Lz in enumerate(Lz_values):
        append_timing(
            "Diabetes",
            M,
            Ni,
            f"Proposed Lz={Lz}",
            len(diab_x_test),
            diab_local_times + diab_private_times[:, lz_index],
        )
        private_mean, private_var = consensus_mean_variance(diab_final[lz_index])
        prediction_rmse = (
            torch.sqrt(
                torch.mean((private_mean - diab_y_test.unsqueeze(0)) ** 2, dim=1)
            )
            .mean()
            .item()
        )
        agreement_rmse = (
            torch.sqrt(
                torch.mean((private_mean - diab_poe_mean.unsqueeze(0)) ** 2, dim=1)
            )
            .mean()
            .item()
        )
        variance_agreement_rmse = (
            torch.sqrt(
                torch.mean((private_var - diab_poe_var.unsqueeze(0)) ** 2, dim=1)
            )
            .mean()
            .item()
        )
        accuracy_rows.append(
            {
                "dataset": "Diabetes",
                "M": M,
                "Ni": Ni,
                "method": f"Proposed Lz={Lz}",
                "prediction_rmse": prediction_rmse,
                "poe_agreement_rmse": agreement_rmse,
                "poe_variance_agreement_rmse": variance_agreement_rmse,
            }
        )

    if M not in nawaz_diabetes_agent_counts:
        run_and_record_nawaz(
            "Diabetes",
            M,
            diab_y_test,
            diab_poe_mean,
            diab_poe_var,
            diab_means,
            diab_vars,
            diab_local_times,
        )
        nawaz_diabetes_agent_counts.add(M)
        owner_hyperparameters = proposed_owner_hyperparameters(
            diab_models, diab_likelihoods
        )
        luo_diabetes_by_M[M] = {
            "owner_hyperparameters": owner_hyperparameters,
            "poe_reference": diab_poe_mean.detach().cpu().clone(),
            "poe_variance_reference": diab_poe_var.detach().cpu().clone(),
        }
        if M == 20:
            luo_diabetes_owner_hyperparameters = owner_hyperparameters

    del sarcos_models, sarcos_likelihoods, diab_models, diab_likelihoods
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

## Diabetes case: 10 agents, four neighbors

Run the additional Diabetes configuration and retain its local GP parameters and PoE reference for the comparison methods.

In [ ]:
M, Ni = 10, 4
print(f'\n{"="*72}\nDiabetes-only case: M={M}, Ni={Ni}\n{"="*72}')
config = dict(base_config)
config.update({"M": M, "Ni": Ni, "k": sarcos_y_shuffled.shape[1]})
graph, wbar = build_graph_and_weights(M, Ni, config["Lw"], seed)
config["Wbar"] = wbar
graph_rows.append(
    {
        "M": M,
        "Ni": Ni,
        "edges": graph.number_of_edges(),
        "connected": True,
        "assumption_2": True,
        "Wbar_integer": True,
    }
)
diab_x_parts = [x.to(device) for x in torch.tensor_split(diab_x_shuffled, M)]
diab_y_parts = [
    y.unsqueeze(-1).to(device) for y in torch.tensor_split(diab_y_shuffled, M)
]

hyper_start = time.perf_counter()
diab_models, diab_likelihoods, _, _, _ = hyperparameter_optimization_1D(
    diab_x_parts,
    diab_y_parts,
    graph,
    config,
    [10.0, 10.0, 20.0],
    device,
    progress_desc="Diabetes M=10, Ni=4 hyperparameters",
    show_progress=True,
)
diab_hyper_time = time.perf_counter() - hyper_start
offline_rows.append(
    {
        "dataset": "Diabetes",
        "M": M,
        "Ni": Ni,
        "hyperparameter_seconds_excluded": diab_hyper_time,
    }
)

diab_means, diab_vars, diab_local_times = benchmark_diabetes_local_prediction(
    diab_models,
    diab_likelihoods,
    diab_x_test,
    num_repeats,
    device,
    workers=model_workers,
)
diab_poe_mean, diab_poe_var, diab_poe_times = benchmark_poe(
    diab_means, diab_vars, num_repeats, "Diabetes PoE aggregation"
)
append_timing(
    "Diabetes",
    M,
    Ni,
    "PoE baseline",
    len(diab_x_test),
    diab_local_times + diab_poe_times,
)
accuracy_rows.append(
    {
        "dataset": "Diabetes",
        "M": M,
        "Ni": Ni,
        "method": "PoE baseline",
        "prediction_rmse": torch.sqrt(
            torch.mean((diab_poe_mean - diab_y_test) ** 2)
        ).item(),
        "poe_agreement_rmse": 0.0,
        "poe_variance_agreement_rmse": 0.0,
    }
)
diab_private_times, diab_final = benchmark_private_consensus(
    diab_means,
    diab_vars,
    graph,
    config,
    Lz_values,
    num_repeats,
    device,
    "Diabetes private consensus",
)
for lz_index, Lz in enumerate(Lz_values):
    append_timing(
        "Diabetes",
        M,
        Ni,
        f"Proposed Lz={Lz}",
        len(diab_x_test),
        diab_local_times + diab_private_times[:, lz_index],
    )
    private_mean, private_var = consensus_mean_variance(diab_final[lz_index])
    accuracy_rows.append(
        {
            "dataset": "Diabetes",
            "M": M,
            "Ni": Ni,
            "method": f"Proposed Lz={Lz}",
            "prediction_rmse": torch.sqrt(
                torch.mean((private_mean - diab_y_test.unsqueeze(0)) ** 2, dim=1)
            )
            .mean()
            .item(),
            "poe_agreement_rmse": torch.sqrt(
                torch.mean((private_mean - diab_poe_mean.unsqueeze(0)) ** 2, dim=1)
            )
            .mean()
            .item(),
            "poe_variance_agreement_rmse": torch.sqrt(
                torch.mean((private_var - diab_poe_var.unsqueeze(0)) ** 2, dim=1)
            )
            .mean()
            .item(),
        }
    )
if M not in nawaz_diabetes_agent_counts:
    run_and_record_nawaz(
        "Diabetes",
        M,
        diab_y_test,
        diab_poe_mean,
        diab_poe_var,
        diab_means,
        diab_vars,
        diab_local_times,
    )
    luo_diabetes_by_M[M] = {
        "owner_hyperparameters": proposed_owner_hyperparameters(
            diab_models, diab_likelihoods
        ),
        "poe_reference": diab_poe_mean.detach().cpu().clone(),
        "poe_variance_reference": diab_poe_var.detach().cpu().clone(),
    }
    nawaz_diabetes_agent_counts.add(M)
del diab_models, diab_likelihoods
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

## Luo-inspired SS-PoE

Two simulated computing servers evaluate local GPs using shared data, PP-Exp, and PP-MI, then aggregate their predictions while they remain shared. Only the final PoE mean and variance are opened.

For each dataset and agent count, reuse the parameters and plaintext reference saved from its first topology (four neighbors). The simulation uses floating-point shares with fixed-point wrapping. Its reported total includes computation, serialization, and modeled communication latency and bandwidth.

In [ ]:
import importlib
import luo_ppgpr.protocol as luo_protocol

importlib.reload(luo_protocol)
FixedPointConfig = luo_protocol.FixedPointConfig
LuoSSPoE = luo_protocol.LuoSSPoE
luo_fixed_point = FixedPointConfig(
    word_bits=64, fractional_bits=26, modulus=base_config["q"]
)
assert set(luo_sarcos_by_M) == {20, 40}
assert set(luo_diabetes_by_M) == {10, 20}


def run_luo_ss_poe(
    dataset,
    owner_count,
    train_x,
    train_y,
    test_x,
    test_y,
    owner_hyperparameters_by_output,
    poe_reference,
    poe_variance_reference,
):
    train_x_np = train_x.detach().cpu().numpy().astype(np.float64)
    train_y_np = train_y.detach().cpu().numpy().astype(np.float64)
    test_x_np = test_x.detach().cpu().numpy().astype(np.float64)
    target = test_y.detach().cpu().numpy().astype(np.float64)
    owner_x_parts = list(np.array_split(train_x_np, owner_count))
    y_matrix = train_y_np[:, None] if train_y_np.ndim == 1 else train_y_np
    total_times, target_rmses = [], []
    protocol_rmses, variance_protocol_rmses = [], []
    final_rounds = None
    for repeat in tqdm(
        range(num_repeats), desc=f"{dataset} Luo-inspired SS-PoE", unit="run"
    ):
        output_predictions, output_variances = [], []
        repeat_time, repeat_rounds = 0.0, 0
        for output, owner_hyperparameters in enumerate(owner_hyperparameters_by_output):
            owner_y_parts = list(np.array_split(y_matrix[:, output], owner_count))
            model = LuoSSPoE(
                fixed_point=luo_fixed_point,
                seed=seed + repeat * owner_count + 1000 * output,
                communication_latency=base_config["communication_latency"],
                communication_bandwidth=625e6,
                simulate_serialization=True,
                owner_workers=num_workers,
            )
            output_mean, output_variance = model.fit_predict(
                owner_x_parts,
                owner_y_parts,
                test_x_np,
                batch_size=len(test_x_np),
                owner_hyperparameters=owner_hyperparameters,
            )
            assert not model.local_predictions_opened
            output_predictions.append(output_mean)
            output_variances.append(output_variance)
            repeat_time += model.latency_adjusted_time
            repeat_rounds += model.online_communication_rounds
        prediction = np.column_stack(output_predictions)
        prediction_variance = np.column_stack(output_variances)
        if train_y_np.ndim == 1:
            prediction = prediction[:, 0]
            prediction_variance = prediction_variance[:, 0]
        target_error = prediction - target
        protocol_error = prediction - poe_reference.detach().cpu().numpy()
        variance_protocol_error = (
            prediction_variance - poe_variance_reference.detach().cpu().numpy()
        )
        target_rmses.append(
            np.sqrt(np.mean(target_error**2))
            if target_error.ndim == 1
            else np.sqrt(np.mean(np.sum(target_error**2, axis=1)))
        )
        protocol_rmses.append(
            np.sqrt(np.mean(protocol_error**2))
            if protocol_error.ndim == 1
            else np.sqrt(np.mean(np.sum(protocol_error**2, axis=1)))
        )
        variance_protocol_rmses.append(
            np.sqrt(np.mean(variance_protocol_error**2))
            if variance_protocol_error.ndim == 1
            else np.sqrt(np.mean(np.sum(variance_protocol_error**2, axis=1)))
        )
        total_times.append(repeat_time)
        final_rounds = repeat_rounds
    append_timing(
        dataset,
        owner_count,
        np.nan,
        "Luo-inspired SS-PoE",
        len(test_x_np),
        total_times,
    )
    accuracy_rows.append(
        {
            "dataset": dataset,
            "M": owner_count,
            "Ni": np.nan,
            "method": "Luo-inspired SS-PoE",
            "prediction_rmse": np.mean(target_rmses),
            "poe_agreement_rmse": np.mean(protocol_rmses),
            "poe_variance_agreement_rmse": np.mean(variance_protocol_rmses),
        }
    )
    print(
        f"{dataset} Luo SS-PoE M={owner_count}: "
        f"target RMSE={np.mean(target_rmses):.6f}; "
        f"PoE mean protocol RMSE={np.mean(protocol_rmses):.12e}; "
        f"variance RMSE={np.mean(variance_protocol_rmses):.12e}; "
        f"total={np.mean(total_times):.6f} +/- {np.std(total_times):.6f} s; "
        f"rounds={final_rounds}"
    )


print("Luo SS-PoE helper ready.")

### SARCOS: 20 and 40 agents

In [ ]:
for M in (20, 40):
    luo_config = luo_sarcos_by_M[M]
    run_luo_ss_poe(
        "SARCOS",
        M,
        sarcos_x_shuffled,
        sarcos_y_shuffled,
        sarcos_x_test,
        sarcos_y_test,
        luo_config["owner_hyperparameters"],
        luo_config["poe_reference"],
        luo_config["poe_variance_reference"],
    )

### Diabetes: 10 and 20 agents

In [ ]:
for M in (10, 20):
    luo_config = luo_diabetes_by_M[M]
    run_luo_ss_poe(
        "Diabetes",
        M,
        diab_x_shuffled,
        diab_y_shuffled,
        diab_x_test,
        diab_y_test,
        luo_config["owner_hyperparameters"],
        luo_config["poe_reference"],
        luo_config["poe_variance_reference"],
    )

## Nawaz-inspired HE-PoE diagnostics

This comparison encrypts the precision-weighted means and precisions from the existing exact local GP predictions. Lattigo performs collective key generation, encryption, addition, and decryption; the aggregate is converted to PoE mean and variance.

The total includes local GP prediction, the backend call, final division, and six 0.2 ms communication phases. This is an HE-PoE adaptation, without sparse GP summaries or separate hyperparameter fitting.

In [ ]:
nawaz_build_table = pd.DataFrame(nawaz_build_rows)
display(
    nawaz_build_table.style.format(
        {
            "local_exact_gp_prediction_seconds_included": "{:.6f}",
            "he_seconds_included": "{:.6f}",
            "agent_division_seconds_included": "{:.6f}",
            "empirical_latency_seconds_included": "{:.6f}",
            "maximum_secure_sum_error": "{:.3e}",
        }
    )
)
assert (nawaz_build_table["kernel"] == "Scaled isotropic RBF").all()

## Graph validation

Check connectivity, Assumption 2, and integer scaled weights for every topology.

In [ ]:
graph_table = pd.DataFrame(graph_rows)
display(graph_table)
assert graph_table["connected"].all()
assert graph_table["assumption_2"].all()
assert graph_table["Wbar_integer"].all()

## Total online runtime

Report seconds for the complete configured test batch, averaged over 20 repeats. Training and data preprocessing are excluded. HE-PoE and SS-PoE are reported once per dataset and agent count with `Ni=N/A`, because their aggregation does not use the neighbor graph.

In [ ]:
timing_table = (
    pd.DataFrame(timing_rows)
    .sort_values(["dataset", "M", "Ni", "method"])
    .reset_index(drop=True)
)
display(
    timing_table.style.format(
        {
            "mean_total_seconds": "{:.6f}",
            "std_total_seconds": "{:.6f}",
            "points_per_second": "{:.2f}",
        }
    )
)
for row in timing_table.itertuples(index=False):
    print(
        f"{row.dataset:8s} | M={int(row.M):2d}, "
        f'Ni={"N/A" if pd.isna(row.Ni) else int(row.Ni)} | '
        f"{row.method:19s} | all {row.test_points:3d} points: "
        f"{row.mean_total_seconds:.6f} +/- {row.std_total_seconds:.6f} s"
    )

## Accuracy

`prediction_rmse` measures error against test targets. The manuscript comparison uses `poe_agreement_rmse` and `poe_variance_agreement_rmse`, which measure mean and variance deviations from the corresponding plaintext PoE reference. Proposed-method errors are averaged over agents. HE-PoE and SS-PoE use the reference saved for their agent count.

In [ ]:
accuracy_table = (
    pd.DataFrame(accuracy_rows)
    .sort_values(["dataset", "M", "Ni", "method"])
    .reset_index(drop=True)
)
display(
    accuracy_table.style.format(
        {
            "prediction_rmse": "{:.6f}",
            "poe_agreement_rmse": "{:.6f}",
            "poe_variance_agreement_rmse": "{:.6f}",
        },
        na_rep="—",
    )
)


poe_agreement_table = accuracy_table[
    accuracy_table["method"].str.startswith("Proposed")
    | accuracy_table["method"].isin(["Nawaz-inspired HE-PoE", "Luo-inspired SS-PoE"])
][
    [
        "dataset",
        "M",
        "Ni",
        "method",
        "poe_agreement_rmse",
        "poe_variance_agreement_rmse",
    ]
].reset_index(
    drop=True
)
display(
    poe_agreement_table.style.format(
        {
            "poe_agreement_rmse": "{:.6f}",
            "poe_variance_agreement_rmse": "{:.6f}",
        },
        na_rep="N/A (graph-independent)",
    )
)

## Offline training time

These diagnostic measurements are excluded from the reported online totals.

In [ ]:
offline_table = (
    pd.DataFrame(offline_rows)
    .sort_values(["dataset", "M", "Ni"])
    .reset_index(drop=True)
)
display(
    offline_table.style.format(
        {
            "hyperparameter_seconds_excluded": "{:.3f}",
        }
    )
)